## Groq LLM Integration

Integrate Groq API for fast LLM-powered responses in the RAG pipeline.

> **About Groq:** Groq provides ultra-fast LLM inference using specialized hardware (LPUs - Language Processing Units). It offers free API access to models like Llama 3, Mixtral, and Gemma with significantly lower latency than traditional GPU-based inference. Ideal for real-time RAG applications.

### Import Libraries

In [1]:
from llama_index.llms.groq import Groq
from llama_index.core import Settings
from dotenv import load_dotenv
import os
import time

### Helper Functions

In [2]:
def estimate_tokens(text):
    """
    Rough estimate of tokens in text (4 chars ≈ 1 token for English).
    
    Args:
        text: Input text string
    
    Returns:
        Estimated token count
    """
    return len(text) // 4


def limit_context(context, max_length=15000, verbose=True):
    """
    Limit context length to avoid Groq API token rate limits.
    
    Args:
        context: The full context string from retrieved documents
        max_length: Maximum character limit (default: 15,000 chars ≈ 3,750 tokens)
        verbose: Whether to print warning when truncating
    
    Returns:
        Truncated context string with notification if truncated
    
    Note:
        Groq free tier has 6,000 TPM (tokens per minute) limit for llama-3.1-8b-instant.
        Keep context under 4,000 tokens to leave room for prompt + response.
    """
    if len(context) > max_length:
        if verbose:
            est_before = estimate_tokens(context)
            est_after = estimate_tokens(context[:max_length])
            print(f"⚠ Context truncated from {len(context):,} to {max_length:,} chars")
            print(f"  (est. tokens: {est_before:,} → {est_after:,} to fit 6K TPM limit)")
        return context[:max_length] + "\n\n[Context truncated to fit token limits]"
    return context


def build_context_from_results(results, max_length=15000):
    """
    Build context from ChromaDB query results with automatic length limiting.
    
    Args:
        results: ChromaDB query results object
        max_length: Maximum character limit for final context
    
    Returns:
        Tuple of (context_string, num_docs)
    """
    context_docs = results["documents"][0]
    context = "\n\n".join(context_docs)
    context = limit_context(context, max_length)
    return context, len(context_docs)

### Configure API Key

> **Note:** Never hardcode API keys in notebooks! Use environment variables:
> 1. Copy `.env.example` to `.env` in the project root
> 2. Add your Groq API key to `.env`
> 3. Load it with `load_dotenv()`
>
> Get your free API key from: https://console.groq.com/keys

In [3]:
# Load environment variables from .env file
load_dotenv()

# Get API key from environment
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY or GROQ_API_KEY == "your_groq_api_key_here":
    print("⚠ Warning: GROQ_API_KEY not set!")
    print("Please create a .env file with your API key.")
    print("See .env.example for template.")
else:
    print("✓ GROQ_API_KEY loaded successfully")
    print(f"  Key starts with: {GROQ_API_KEY[:8]}...")

✓ GROQ_API_KEY loaded successfully
  Key starts with: gsk_Q5Ti...


### Initialize Groq LLM

In [4]:
# Define model configuration
# Recommended models for free tier (better rate limits):
# - llama-3.1-8b-instant: Fast, efficient, 128K context (BEST for free tier)
# - llama-3.2-11b-vision-preview: Good balance, 128K context
# - gemma2-9b-it: Good quality, 8K context
# Avoid for free tier: llama-3.3-70b-versatile (hits rate limits quickly)
MODEL_NAME = "llama-3.1-8b-instant"
CONTEXT_WINDOW = 131072  # 128K context support

# Initialize Groq LLM
llm = Groq(
    model=MODEL_NAME,
    api_key=GROQ_API_KEY,
    context_window=CONTEXT_WINDOW
)

# Set as global LLM for LlamaIndex
Settings.llm = llm

print(f"✓ Groq LLM initialized: {MODEL_NAME}")
print(f"  Context window: {CONTEXT_WINDOW} tokens")

✓ Groq LLM initialized: llama-3.1-8b-instant
  Context window: 131072 tokens


### Test Basic Completion

In [5]:
# Simple test query
test_query = "What is a contract in legal terms?"

# Generate completion
start_time = time.time()
response = llm.complete(test_query)
elapsed = time.time() - start_time

print(f"Query: {test_query}\n")
print(f"Response:\n{response}")
print(f"\n⏱ Response time: {elapsed:.2f} seconds")

Query: What is a contract in legal terms?

Response:
In legal terms, a contract is a binding agreement between two or more parties that creates, modifies, or extinguishes a legal relationship. It is a written or oral agreement that is enforceable by law, and it outlines the terms and conditions of the agreement, including the rights and obligations of each party.

A contract typically consists of the following essential elements:

1. **Offer**: One party makes a proposal to the other party, which is a clear and specific expression of their intention to enter into an agreement.
2. **Acceptance**: The other party agrees to the terms of the offer, which is a clear and unambiguous expression of their acceptance.
3. **Consideration**: Both parties must provide something of value, such as money, goods, or services, in exchange for the other party's performance.
4. **Capacity**: Both parties must have the legal capacity to enter into a contract, meaning they must be of sound mind, not a minor

### Streaming Responses

> **Why Streaming?** For long responses, streaming shows tokens as they're generated, improving user experience by reducing perceived latency.

In [6]:
# Test streaming response
stream_query = "Explain the difference between civil law and criminal law in 3-4 sentences."

print(f"Query: {stream_query}\n")
print("Streaming response:")
print("-" * 50)

start_time = time.time()

# Stream tokens as they arrive
for chunk in llm.stream_complete(stream_query):
    print(chunk.delta, end="")

elapsed = time.time() - start_time
print(f"\n-" * 50)
print(f"⏱ Streaming completed in {elapsed:.2f} seconds")

Query: Explain the difference between civil law and criminal law in 3-4 sentences.

Streaming response:
--------------------------------------------------
Civil law deals with disputes between individuals, organizations, or government entities, focusing on resolving conflicts and awarding compensation for damages or injuries. It typically involves lawsuits between private parties, such as personal injury claims, contract disputes, or property rights. In contrast, criminal law involves the prosecution of individuals or organizations for violating laws that protect society, with the primary goal of punishing offenders and maintaining public safety. Criminal law is enforced by the state, whereas civil law is typically resolved through private litigation.
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
⏱ Streaming completed in 0.64 seconds


### Compare Different Models

In [7]:
# Available Groq models - Recommended for free tier (as of 2026)
# Check https://console.groq.com/docs/models for latest
# Priority: Smaller models with better rate limits
MODELS = [
    "llama-3.1-8b-instant",       # BEST for free tier - Fast & efficient, 6K TPM
    "llama-3.3-70b-versatile",    # Higher quality, but hits rate limits faster
]

test_prompt = "What is habeas corpus?"

print(f"Testing models with: {test_prompt}\n")
print("=" * 60)
print("Note: Smaller models (8B) have better TPM limits\n")
print("Tip: If you hit rate limits, stick with llama-3.1-8b-instant\n")
print("=" * 60)

for model_name in MODELS:
    print(f"\nModel: {model_name}")
    print("-" * 40)
    
    try:
        # Create temporary LLM with this model
        test_llm = Groq(model=model_name, api_key=GROQ_API_KEY)
        
        # Measure response time
        start = time.time()
        response = test_llm.complete(test_prompt)
        elapsed = time.time() - start
        
        print(f"Response: {str(response)[:150]}...")
        print(f"⏱ Time: {elapsed:.2f}s")
    except Exception as e:
        print(f"Error: {e}")

Testing models with: What is habeas corpus?

Note: Smaller models (8B) have better TPM limits

Tip: If you hit rate limits, stick with llama-3.1-8b-instant


Model: llama-3.1-8b-instant
----------------------------------------
Response: Habeas corpus is a Latin phrase that translates to "you have the body." It is a fundamental legal concept in many countries, particularly in common la...
⏱ Time: 0.99s

Model: llama-3.3-70b-versatile
----------------------------------------
Response: Habeas corpus is a fundamental legal principle that protects individuals from unlawful detention or imprisonment. It is a Latin phrase that translates...
⏱ Time: 1.71s


### Load Document Chunks from ChromaDB

In [8]:
# Connect to ChromaDB (from Notebook 05)
import chromadb
from sentence_transformers import SentenceTransformer
from pathlib import Path

# Global model constant
MODEL_NAME = "all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME)

# Connect to Docker ChromaDB
chroma_client = chromadb.HttpClient(host="localhost", port="8000")

# Get collection
collection = chroma_client.get_or_create_collection(name="legal-docs")

print(f"✓ Connected to ChromaDB collection: {collection.name}")
print(f"  Total documents: {collection.count()}")

✓ Connected to ChromaDB collection: legal-docs
  Total documents: 4


### Retrieve Relevant Context

In [9]:
# Query to search for
query = "What are the forest conservation issues?"

# Generate query embedding
query_embedding = model.encode([query]).tolist()

# Retrieve top 1 document to stay within TPM limits
# Note: Groq free tier has 6,000 TPM (tokens per minute) limit for llama-3.1-8b-instant
# Increase n_results only if you have higher TPM limits
results = collection.query(
    query_embeddings=query_embedding,
    n_results=1,  # Reduced to 1 to fit 6K TPM limit
    include=["documents", "metadatas", "distances"]
)

# Use helper function to build and limit context (15K chars ≈ 3,750 tokens)
context, num_docs = build_context_from_results(results, max_length=15000)

print(f"Query: {query}\n")
print(f"Retrieved {num_docs} relevant documents")
print(f"Context length: {len(context):,} characters")
print(f"\n⚠ Note: Groq free tier has 6,000 TPM limit. Using 1 doc + 15K char limit.")

⚠ Context truncated from 20,684 to 15,000 chars
  (est. tokens: 5,171 → 3,750 to fit 6K TPM limit)
Query: What are the forest conservation issues?

Retrieved 1 relevant documents
Context length: 15,041 characters

⚠ Note: Groq free tier has 6,000 TPM limit. Using 1 doc + 15K char limit.


### Generate Answer with Context (RAG)

In [10]:
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from openai import APIStatusError


def is_rate_limit_error(exception):
    """Check if exception is a rate limit error (429 or 413)."""
    if isinstance(exception, APIStatusError):
        return exception.status_code in [429, 413]
    return False


@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=5, max=60),
    retry=retry_if_exception_type(APIStatusError),
    reraise=True
)
def llm_complete_with_retry(llm, prompt, max_retries=3):
    """
    Call LLM completion with retry logic for rate limit errors.
    
    Args:
        llm: Groq LLM instance
        prompt: Input prompt
        max_retries: Maximum number of retry attempts
    
    Returns:
        LLM completion response
    
    Note:
        Retries on 429 (rate limit) and 413 (request too large) errors
        with exponential backoff: 5s, 10s, 20s, etc.
    """
    try:
        return llm.complete(prompt)
    except APIStatusError as e:
        if is_rate_limit_error(e):
            print(f"⚠ Rate limit hit: {e}")
            print(f"  Retrying with exponential backoff...")
        raise


def llm_stream_complete_with_retry(llm, prompt):
    """
    Call LLM streaming completion with retry logic for rate limit errors.
    
    Args:
        llm: Groq LLM instance
        prompt: Input prompt
    
    Yields:
        Streaming chunks from LLM
    
    Note:
        Retries on 429 (rate limit) and 413 (request too large) errors
        with exponential backoff: 5s, 10s, 20s (up to 3 retries)
    """
    max_retries = 3
    base_wait = 5  # seconds
    
    for attempt in range(max_retries):
        try:
            for chunk in llm.stream_complete(prompt):
                yield chunk
            return  # Success, exit the function
        except APIStatusError as e:
            if is_rate_limit_error(e) and attempt < max_retries - 1:
                wait_time = base_wait * (2 ** attempt)
                print(f"\n⚠ Rate limit hit (attempt {attempt + 1}/{max_retries})")
                print(f"  Waiting {wait_time}s before retry...")
                time.sleep(wait_time)
            else:
                raise


# RAG prompt template
RAG_PROMPT = """
You are a legal document assistant. Use the provided context to answer the question.

Context:
{context}

Question: {question}

Instructions:
1. Answer ONLY based on the provided context
2. Cite specific sections or sources when possible
3. If the answer is not in the context, say "I don't have enough information"
4. Keep responses concise but informative

Answer:
"""

# Format prompt with context and question
formatted_prompt = RAG_PROMPT.format(context=context, question=query)

# Estimate tokens before calling API
est_tokens = estimate_tokens(formatted_prompt)
print(f"Estimated prompt size: {est_tokens:,} tokens")
print(f"(Groq free tier limit: 6,000 TPM for llama-3.1-8b-instant)\n")

# Generate response with retry logic
start_time = time.time()
response = llm_complete_with_retry(llm, formatted_prompt)
elapsed = time.time() - start_time

print(f"Question: {query}\n")
print(f"Answer:\n{response}")
print(f"\n⏱ Response time: {elapsed:.2f} seconds")

Estimated prompt size: 3,857 tokens
(Groq free tier limit: 6,000 TPM for llama-3.1-8b-instant)

Question: What are the forest conservation issues?

Answer:
The forest conservation issues in the provided context are:

1. **Encroachment of forest areas**: The report submitted by the Ministry of Environment and Forests indicates that almost 13,000 sq. km of forest area is under encroachment (para 11).
2. **Depletion of forest cover**: The world is facing calamities caused by climate change, and the primary culprit behind this is the depleting forest cover due to rapid urbanization, unchecked industrialization, encroachments, etc. (para 5).
3. **Restoration of degraded forest areas**: The High Court of Madras was requested to restore the degraded forest areas, but the issue was left inconclusive (para 3).
4. **Protection of forests**: The Supreme Court has repeatedly issued mandatory directions to the States and other authorities to ensure that the forest cover is maintained/restored and a

### RAG with Streaming

In [11]:
# Another query with streaming response
query2 = "What is the significance of the Simlipal Tiger Reserve?"

# Get context (using helper function to avoid token limits)
query_embedding = model.encode([query2]).tolist()
results = collection.query(
    query_embeddings=query_embedding,
    n_results=1,  # Reduced to 1 to fit 6K TPM limit
    include=["documents"]
)

# Use helper function to build and limit context (15K chars ≈ 3,750 tokens)
context, num_docs = build_context_from_results(results, max_length=15000)

# Format prompt
formatted_prompt = RAG_PROMPT.format(context=context, question=query2)

# Estimate tokens before calling API
est_tokens = estimate_tokens(formatted_prompt)
print(f"Question: {query2}\n")
print(f"Using {num_docs} documents (context: {len(context):,} chars)")
print(f"Estimated prompt size: {est_tokens:,} tokens")
print(f"(Groq free tier limit: 6,000 TPM for llama-3.1-8b-instant)\n")
print("Streaming Answer:")
print("-" * 50)

start_time = time.time()
try:
    for chunk in llm_stream_complete_with_retry(llm, formatted_prompt):
        print(chunk.delta, end="")
    elapsed = time.time() - start_time
    print(f"\n-" * 50)
    print(f"⏱ Streaming completed in {elapsed:.2f} seconds")
except APIStatusError as e:
    print(f"\n\n⚠ Error: {e}")
    if e.status_code in [429, 413]:
        print("\n💡 Tip: This is a rate limit error. Try:")
        print("   1. Wait a minute and retry (TPM limit resets)")
        print("   2. Reduce n_results to 1")
        print("   3. Reduce max_length to 10000")

⚠ Context truncated from 20,684 to 15,000 chars
  (est. tokens: 5,171 → 3,750 to fit 6K TPM limit)
Question: What is the significance of the Simlipal Tiger Reserve?

Using 1 documents (context: 15,041 chars)
Estimated prompt size: 3,861 tokens
(Groq free tier limit: 6,000 TPM for llama-3.1-8b-instant)

Streaming Answer:
--------------------------------------------------
I don't have enough information about the Simlipal Tiger Reserve in the provided context. The context mentions the Kalakkad-Mundanthurai Tiger Reserve and the Agasthyamalai Biosphere Reserve, but not the Simlipal Tiger Reserve.
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
⏱ Streaming completed in 11.82 seconds


### Test Multiple Queries

In [12]:
# Test multiple legal queries
queries = [
    "What are the grounds for appeal?",
    "What is the Supreme Court's reasoning?",
    "What penalties are mentioned?"
]

print("Testing multiple RAG queries:\n")
print("=" * 60)

for i, q in enumerate(queries, 1):
    print(f"\n{i}. Query: {q}")
    
    # Retrieve context (1 doc to fit TPM limits)
    q_embedding = model.encode([q]).tolist()
    results = collection.query(
        query_embeddings=q_embedding,
        n_results=1,  # Reduced to fit 6K TPM limit
        include=["documents"]
    )
    
    # Use helper function to build and limit context (15K chars)
    context, num_docs = build_context_from_results(results, max_length=15000)
    
    # Generate answer with retry logic
    prompt = RAG_PROMPT.format(context=context, question=q)
    est_tokens = estimate_tokens(prompt)
    print(f"   Context: {num_docs} doc(s), ~{est_tokens:,} tokens")
    
    try:
        response = llm_complete_with_retry(llm, prompt)
        print(f"   Answer: {str(response)[:200]}...")
    except APIStatusError as e:
        print(f"   ⚠ Error: {e}")
        if e.status_code in [429, 413]:
            print("   💡 Skipping - rate limit reached. Wait and retry.")

Testing multiple RAG queries:


1. Query: What are the grounds for appeal?
⚠ Context truncated from 26,467 to 15,000 chars
  (est. tokens: 6,616 → 3,750 to fit 6K TPM limit)
   Context: 1 doc(s), ~3,855 tokens
   Answer: According to the context, the grounds for appeal against an order approving a resolution plan under Section 31 of the Insolvency and Bankruptcy Code (IBC) are as follows:

(i) The approved resolution ...

2. Query: What is the Supreme Court's reasoning?
⚠ Context truncated from 25,181 to 15,000 chars
  (est. tokens: 6,295 → 3,750 to fit 6K TPM limit)
   Context: 1 doc(s), ~3,857 tokens
   Answer: The Supreme Court's reasoning is as follows:

The Court held that the appeals were barred by limitation and dismissed them. The reasoning is based on Section 61 of the Insolvency and Bankruptcy Code (...

3. Query: What penalties are mentioned?
⚠ Context truncated from 26,467 to 15,000 chars
  (est. tokens: 6,616 → 3,750 to fit 6K TPM limit)
   Context: 1 doc(s), ~3,854 tokens

### Performance Statistics

In [14]:
# Performance benchmark
import numpy as np

test_queries = [
    "What is environmental law?",
    "Explain forest conservation",
    "What are tribal rights?"
]

response_times = []
failed_queries = []

print("Running performance benchmark...\n")
print("=" * 60)
print("Note: Using max_length=15000 to fit Groq 6K TPM limit")
print("=" * 60)

for q in test_queries:
    q_embedding = model.encode([q]).tolist()
    results = collection.query(
        query_embeddings=q_embedding,
        n_results=1,  # Reduced to 1 to fit 6K TPM limit
        include=["documents"]
    )
    
    # Use helper function to build and limit context (avoids 413 errors)
    # max_length=15000 chars ≈ 3,750 tokens (well under 6K limit)
    context, num_docs = build_context_from_results(results, max_length=15000)
    prompt = RAG_PROMPT.format(context=context, question=q)
    
    # Estimate tokens before calling API
    est_tokens = estimate_tokens(prompt)
    print(f"\nQuery: {q[:50]}...")
    print(f"  Context: {num_docs} doc(s), ~{est_tokens:,} tokens")
    
    try:
        start = time.time()
        _ = llm_complete_with_retry(llm, prompt)
        elapsed = time.time() - start
        
        response_times.append(elapsed)
        print(f"  ✓ Response time: {elapsed:.2f}s")
    except APIStatusError as e:
        print(f"  ⚠ Error: {e}")
        if e.status_code in [429, 413]:
            print(f"  💡 Rate limit hit - skipping this query")
            failed_queries.append((q, str(e)))
        else:
            raise

print(f"\n{'=' * 60}")
print("Performance Summary:")
print(f"  Model: {MODEL_NAME}")
print(f"  LLM: {llm.model}")
print(f"  Total queries: {len(test_queries)}")
print(f"  Successful: {len(response_times)}")
print(f"  Failed: {len(failed_queries)}")

if response_times:
    print(f"  Average response time: {np.mean(response_times):.2f}s")
    print(f"  Min response time: {np.min(response_times):.2f}s")
    print(f"  Max response time: {np.max(response_times):.2f}s")
    print(f"  Std deviation: {np.std(response_times):.2f}s")

if failed_queries:
    print(f"\n⚠ Failed Queries:")
    for q, err in failed_queries:
        print(f"  - {q[:50]}...: {err[:100]}")

Running performance benchmark...

Note: Using max_length=15000 to fit Groq 6K TPM limit
⚠ Context truncated from 20,684 to 15,000 chars
  (est. tokens: 5,171 → 3,750 to fit 6K TPM limit)

Query: What is environmental law?...
  Context: 1 doc(s), ~3,854 tokens
  ✓ Response time: 1.95s
⚠ Context truncated from 20,684 to 15,000 chars
  (est. tokens: 5,171 → 3,750 to fit 6K TPM limit)

Query: Explain forest conservation...
  Context: 1 doc(s), ~3,854 tokens
  ✓ Response time: 11.53s
⚠ Context truncated from 21,587 to 15,000 chars
  (est. tokens: 5,396 → 3,750 to fit 6K TPM limit)

Query: What are tribal rights?...
  Context: 1 doc(s), ~3,853 tokens
  ✓ Response time: 37.81s

Performance Summary:
  Model: all-MiniLM-L6-v2
  LLM: llama-3.1-8b-instant
  Total queries: 3
  Successful: 3
  Failed: 0
  Average response time: 17.10s
  Min response time: 1.95s
  Max response time: 37.81s
  Std deviation: 15.16s


### Token Usage Estimation

In [15]:
# Estimate token usage (approximate: 1 token ≈ 4 characters)
def estimate_tokens(text):
    return len(text) / 4

# Analyze last RAG interaction
prompt_tokens = estimate_tokens(formatted_prompt)
response_tokens = estimate_tokens(str(response))
total_tokens = prompt_tokens + response_tokens

print("Token Usage Estimate:")
print(f"  Input (prompt + context): {prompt_tokens:.0f} tokens")
print(f"  Output (response): {response_tokens:.0f} tokens")
print(f"  Total: {total_tokens:.0f} tokens")
print(f"\n  Note: Groq free tier includes ~14,400 requests/day (check console for latest limits)")

Token Usage Estimate:
  Input (prompt + context): 3861 tokens
  Output (response): 225 tokens
  Total: 4086 tokens

  Note: Groq free tier includes ~14,400 requests/day (check console for latest limits)
